# IZQAR IBM AML · validation-only causal history shadow export

**No final test / no production deployment.** This notebook re-trains the frozen IBM `causal_history_v1` candidate on the original synthetic HI-Small CSV, validates on the next chronological period, and optionally exports a SHA-pinned, offline-only shadow bundle. An earlier report-only ZIP does not contain weights, so it cannot be converted to a trained model.

Add Kaggle Input: `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`. Before starting, publish the IZQAR `feat/ibm-shadow-export` changes to GitHub, then paste their **exact 40-character SHA** below.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile

EXPECTED_REVISION = 'REPLACE_WITH_40_CHAR_COMMIT_SHA'
if len(EXPECTED_REVISION) != 40 or any(c not in '0123456789abcdef' for c in EXPECTED_REVISION):
    raise RuntimeError('PIN EXACT GIT SHA before starting: this notebook refuses unreviewed code')
root = Path('/kaggle/working/izqar_ibm_shadow')
if root.exists():
    raise FileExistsError('Fresh Kaggle working directory required; refusing mixed revisions')
subprocess.run(['git', 'clone', '--branch', 'feat/ibm-shadow-export',
                'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=root, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
if actual != EXPECTED_REVISION:
    raise RuntimeError('Pinned code revision does not match reviewed code')
backend = root / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(backend / 'requirements-ml.txt'), 'scikit-learn==1.9.1'], check=True)
candidates = [p for p in Path('/kaggle/input').rglob('HI-Small_Trans.csv')
              if 'ibm-transactions-for-anti-money-laundering-aml' in str(p)]
if len(candidates) != 1:
    raise FileNotFoundError(f'Expected exactly one IBM HI-Small CSV from pinned Kaggle input; found {len(candidates)}')
data = candidates[0]
print('Revision', actual, 'dataset', data.name, 'bytes', data.stat().st_size)


In [ ]:
output = Path('/kaggle/working/ibm_causal_shadow_v1.joblib')
bundle = Path('/kaggle/working/ibm_causal_shadow_v1.zip')
if output.exists() or output.with_suffix('.metrics.json').exists() or bundle.exists():
    raise FileExistsError('Experiment output already exists; do not overwrite another result')
subprocess.run([sys.executable, str(backend / 'scripts/train_ibm_aml.py'),
                '--data', str(data), '--mode', 'experiment',
                '--feature-set', 'causal_history_v1', '--full-train',
                '--estimators', '180', '--max-false-alerts-per-10000', '100',
                '--output', str(output), '--shadow-bundle', str(bundle)],
               cwd=backend, check=True)
report = json.loads(output.with_suffix('.metrics.json').read_text(encoding='utf-8'))
assert report['run_mode'] == 'experiment'
assert report['test_metrics'] is None and report['baseline_test_metrics'] is None
assert not output.exists()  # Never produces a serving joblib artifact.
sys.path.insert(0, str(backend / 'scripts'))
from ibm_shadow_bundle import verify_shadow_bundle
verified = verify_shadow_bundle(bundle)  # No unpickling during verification.
assert verified['validation_metrics'] == report['validation_metrics']
sha256 = verified['sha256']
print(json.dumps({'revision': actual, 'dataset_bytes': data.stat().st_size,
                  'report': str(output.with_suffix('.metrics.json')),
                  'bundle': str(bundle), 'bundle_sha256': sha256,
                  'validation': report['validation_metrics'],
                  'warning': 'Research candidate. NO sealed test, NO API promotion.'}, indent=2))


## Review before download

Compare the **new validation metrics** to the documented baseline on the same chronological window. Preserve the exact SHA256 displayed above in a separate log and only then download the ZIP from Kaggle Output. Do **not** commit the ZIP or its preprocessor/weights to Git. Do **not** set the active `IZQAR_IBM_AML_MODEL_PATH` to these files.

Run the local verifier/importer as documented in `docs/IBM_SHADOW_WORKFLOW.md`.
